# Exploration Notebook to test different scraping methods

Goal: Scrape basic information from Berlin, Hamburg, Munich off of Wikipedia:
1. Country
2. Latitude and longitude
3. Population

In [8]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

Step 1: load HTML of respective sites

In [ ]:
url_berlin = "https://en.wikipedia.org/wiki/Berlin"
url_hamburg = "https://en.wikipedia.org/wiki/Hamburg"
url_munich = "https://en.wikipedia.org/wiki/Munich"
headers = {'User-Agent': 'Chrome/134.0.0.0'}

response_berlin = requests.get(url_berlin, headers=headers)
response_hamburg = requests.get(url_hamburg, headers=headers)
response_munich = requests.get(url_munich, headers=headers)

soup_berlin = BeautifulSoup(response_berlin.content, 'html.parser')
soup_hamburg = BeautifulSoup(response_hamburg.content, 'html.parser')
soup_munich = BeautifulSoup(response_munich.content, 'html.parser')

In [ ]:
url_dict = {"Berlin": "https://en.wikipedia.org/wiki/Berlin",
            "Hamburg": "https://en.wikipedia.org/wiki/Hamburg",
            "Munich": "https://en.wikipedia.org/wiki/Munich"}
headers = {'User-Agent': 'Chrome/134.0.0.0'}

soup_dict = {}

for city, url in url_dict.items():
    soup = BeautifulSoup(requests.get(url, headers=headers).content, 'html.parser')
    soup_dict[city] = soup

In [57]:
# function to parse latitude and longitude

def parse_dms(coord):
    pattern = r"(\d+)°(\d+)′(?:(\d+)″)?([NSEW])"
    match = re.match(pattern, coord)
    
    if not match:
        raise ValueError(f"Invalid coordinate: {coord}")
    
    deg = int(match.group(1))
    minutes = int(match.group(2))
    seconds = match.group(3)
    direction = match.group(4)
    
    seconds = int(seconds) if seconds else 0
    
    decimal = deg + minutes / 60 + seconds / 3600
    
    if direction in "SW":
        decimal = -decimal
        
    return decimal

In [59]:
data_frame_rows = []

for city, soup in soup_dict.items():
    # country
    info_box_country = soup.find(class_ = 'infobox-label', string=re.compile("country", re.IGNORECASE)) 
    country = info_box_country.find_next().get_text()
    # latitude
    latitude = round(parse_dms(soup.find(class_ = 'latitude').get_text()), 5)
    # longitude
    longitude = round(parse_dms(soup.find(class_ = 'longitude').get_text()), 5)
    # population
    pop_rows = soup.select("table.infobox tr")
    for i, row in enumerate(pop_rows):
        if "Population" in row.get_text():
            population = pop_rows[i+1].find("td").get_text(strip=True)
            break

    data_frame_rows.append({
        "city": city,
        "country": country,
        "latitude": latitude,
        "longitude": longitude,
        "population": population
        })

data_frame_rows

[{'city': 'Berlin',
  'country': 'Germany',
  'latitude': 52.52,
  'longitude': 13.405,
  'population': '3,596,999'},
 {'city': 'Hamburg',
  'country': 'Germany',
  'latitude': 53.55,
  'longitude': 10.0,
  'population': '1,973,896'},
 {'city': 'Munich',
  'country': 'Germany',
  'latitude': 48.1375,
  'longitude': 11.575,
  'population': '1,505,005'}]

In [51]:
city_infos = pd.DataFrame(data_frame_rows)
city_infos

,city,country,latitude,longitude,population
0,Berlin,Germany,52°31′12″N,13°24′18″E,"3,596,999"
1,Hamburg,Germany,53°33′N,10°00′E,"1,973,896"
2,Munich,Germany,48°08′15″N,11°34′30″E,"1,505,005"
